# Sentinel-2 L1C subsets around the AERONET-OC stations

Image dataset for the validation of the GRS L2A remote sensing reflectance with AERONET-OC: Sentinel-2 L1C
subsets of **5 km x 5 km** (251 x 251 pixels at 20 m) centred on the AERONET-OC stations, for the overpasses
with an AERONET-OC measurement (Level 1.5 or 2.0) within **2 h**, in a local S3 bucket with a static
**STAC catalogue**.

Each subset is the product of GRSdriver (`Sentinel2Driver`): TOA reflectance of the 13 bands at 20 m, sun angles,
**viewing angles computed per band and per detector** (at 60 m, resampled), spectral responses (SRF). It is
written as NetCDF with the bands and angles packed in int16 (about 1.3 MB) and is the direct input of GRS
(`grs <subset>.nc`, about 25 s per image instead of minutes on a full tile; the angles are not recomputed).

| item | value |
|---|---|
| stations | sites of the validation database (`/data/insitu/validation_db/harmonised/aeronet_oc.nc`); the sites at the same position are one station (Venise + AAOT, USC_SEAPRISM + USC_SEAPRISM_2) |
| images | CDSE catalogue, last processing of each image; one tile per datatake (best coverage of the box, then an image already on this machine, then the UTM zone of the station); the station inside the data footprint |
| source | images of this machine (`/data/satellite/Sentinel-2/L1C`, same datatake, any baseline) read in place; the others read in place in the CDSE S3 bucket (only the parts of the files in the box), or downloaded, subset and deleted |
| bucket | `s3://aeronet-oc`: content in `/data/satellite/validation`, linked from `/data/satellite/s3/aeronet-oc` (served with the other buckets by `grstbx-s3-bucket`); plans, logs and downloads in `/data/satellite/validation/work` (not in the bucket) |
| subsets | `s3://aeronet-oc/sentinel-2/l1c/<station>/YYYY/MM/DD/<name>.nc`, `<name>.json` (STAC Item, written last), `<name>_ql.png`; GRS L2A products in `sentinel-2/l2a/` |
| catalogue | `s3://aeronet-oc/stac/catalog.json`, collection `s2-l1c-aeronet-oc` (`items.json`: all the Items) |
| stations | `s3://aeronet-oc/stations.geojson` |

The dataset is made by `aeronet_oc_s2_subsets.py` (folder of this notebook):

```bash
python aeronet_oc_s2_subsets.py stations                  # stations of the validation database
python aeronet_oc_s2_subsets.py all --workers 4 --source remote   # plan + subsets, station after station
python aeronet_oc_s2_subsets.py run --stations AAOT --retry-failed
python aeronet_oc_s2_subsets.py catalog                   # STAC catalogue (also rebuilt after each station)
```

It can be stopped and restarted at any time: the subsets with their Item are skipped. The GRS L2A products
of the subsets and their matchups with AERONET-OC: `grs_l2a_aeronet_oc_matchups.ipynb`.

In [ ]:
# development versions of the GRS suite (repositories), before the installed packages, also for the
# processes started by this notebook; remove to use the installed versions (GRS >= 3.1.1, GRSdriver >= 2.1.1)
import os, sys
DEV_REPOS = [f'/home/harmel/Dropbox/Dropbox/work/git/satellite_app/{repo}'
             for repo in ('grs', 'GRSl2bgen', 'GRSdriver', 'grstbx')]
DEV_REPOS = [repo for repo in DEV_REPOS if os.path.isdir(repo)]
sys.path[:0] = DEV_REPOS
os.environ['PYTHONPATH'] = os.pathsep.join(DEV_REPOS + [os.environ.get('PYTHONPATH', '')]).strip(os.pathsep)

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

from grstbx.app import search_items

DATA_ROOT = Path('/data/satellite/validation')              # content of the bucket s3://aeronet-oc
CATALOG = DATA_ROOT / 'stac' / 'catalog.json'
ITEMS = DATA_ROOT / 'stac' / 's2-l1c-aeronet-oc' / 'items.json'
L1C_DIR = DATA_ROOT / 'sentinel-2' / 'l1c'

## 1. Stations

`stations.geojson`: position of each station, its AERONET-OC sites and measurements since July 2015, the number of
subsets in the bucket and the box of the subsets (lon/lat).

In [ ]:
stations = json.loads((DATA_ROOT / 'stations.geojson').read_text())
st = pd.DataFrame([{**f['properties'], 'lon': f['geometry']['coordinates'][0], 'lat': f['geometry']['coordinates'][1]}
                   for f in stations['features']]).set_index('station')
print(f"{len(st)} stations, {st.n_s2_subsets.sum()} subsets")
st[['sites', 'lon', 'lat', 'n_aeronet_obs', 'first', 'last', 'n_s2_subsets']]

In [ ]:
fig, ax = plt.subplots(figsize=(13, 6))
try:                                       # coastlines if cartopy and its Natural Earth data are available
    import cartopy.feature as cfeature
    import cartopy.crs as ccrs
    plt.close(fig)
    fig, ax = plt.subplots(figsize=(13, 6), subplot_kw={'projection': ccrs.PlateCarree()})
    ax.add_feature(cfeature.LAND, color='0.9'); ax.coastlines(lw=0.4); ax.set_global()
    kw = {'transform': ccrs.PlateCarree()}
except Exception:
    kw = {}
sc = ax.scatter(st.lon, st.lat, s=10 + st.n_s2_subsets / 3, c=st.n_aeronet_obs, cmap='viridis', **kw)
for name, row in st.iterrows():
    ax.text(row.lon, row.lat, f' {name}', fontsize=6, **kw)
plt.colorbar(sc, ax=ax, label='AERONET-OC measurements since 2015-07', shrink=0.6)
ax.set_title('AERONET-OC stations (marker size: number of Sentinel-2 subsets)');

## 2. STAC catalogue

{func}`grstbx.app.search_items` searches the static catalogue (local folder or bucket URL) by area, period and
cloud cover, as in the dashboard of grstbx (`grstbx-viewer --stac /data/satellite/validation/stac/catalog.json`).
The properties of the Items give the AERONET-OC measurements around the overpass: `aeronet_oc:n_obs`
(within 2 h), `aeronet_oc:n_obs_lev20` (Level 2.0), `aeronet_oc:dt_min` (time of the nearest measurement
minus overpass, minutes).

In [ ]:
found = search_items(str(CATALOG), start='2015-07-01', end='2026-12-31')
print(len(found), 'subsets in the catalogue')
found[['time', 'tile', 'cloud_cover', 'path']].tail()

In [ ]:
# all the properties of the Items, read at once from items.json
items = json.loads(ITEMS.read_text())['features']
props = pd.DataFrame([{'id': item['id'], **item['properties'], 'href': item['assets']['data']['href']} for item in items])
props['datetime'] = pd.to_datetime(props['datetime'])
summary = props.groupby('aeronet_oc:station').agg(
    subsets=('id', 'size'), with_lev20=('aeronet_oc:n_obs_lev20', lambda n: int((n > 0).sum())),
    within_1h=('aeronet_oc:dt_min', lambda dt: int((dt.abs() <= 60).sum())),
    first=('datetime', 'min'), last=('datetime', 'max'), tiles=('grid:code', lambda g: sorted(set(g))))
summary.sort_values('subsets', ascending=False)

## 3. A subset

The subsets are read with xarray (`decode_coords='all'`: CRS of the tile in `spatial_ref`). Variables: `bands`
(TOA reflectance, `wl` x `y` x `x`), `vza` and `raa` (viewing zenith and relative azimuth, per band), `sza`,
`SRF` (spectral responses on `wl_hr`), `wl_true` (central wavelengths); attributes: the metadata of the L1C
product (`MTD_MSIL1C.xml`, `MTD_TL.xml`).

In [ ]:
STATION = 'AAOT'
sel = props[props['aeronet_oc:station'] == STATION].sort_values('datetime')
# a clear image: a Level 2.0 measurement within 30 min
row = sel[(sel['aeronet_oc:n_obs_lev20'] > 0) & (sel['aeronet_oc:dt_min'].abs() < 30)].iloc[-1] if len(sel) else None
path = L1C_DIR / row.href.split('/sentinel-2/l1c/')[1]
ds = xr.open_dataset(path, decode_coords='all')
print(path.name, ds.rio.crs, dict(ds.sizes))
ds

In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(18, 4.2))
rgb = ds.bands.sel(wl=[665, 560, 490])
(rgb / 0.25).clip(0, 1).plot.imshow(rgb='wl', ax=axs[0])
axs[0].plot(ds.x[ds.sizes['x'] // 2], ds.y[ds.sizes['y'] // 2], 'r+', ms=12)
axs[0].set_title('TOA RGB, station at the centre')
ds.vza.sel(wl=865).plot(ax=axs[1], cmap='viridis'); axs[1].set_title('vza (865 nm), per detector')
ds.raa.sel(wl=865).plot(ax=axs[2], cmap='twilight'); axs[2].set_title('raa (865 nm)')
centre = ds.isel(x=ds.sizes['x'] // 2, y=ds.sizes['y'] // 2)
axs[3].plot(ds.wl, centre.bands, 'o-')
axs[3].set(xlabel='wavelength (nm)', ylabel='TOA reflectance', title='Pixel of the station')
for ax in axs[:3]:
    ax.set_aspect('equal'); ax.set_xticks([]); ax.set_yticks([]); ax.set_xlabel(''); ax.set_ylabel('')
fig.tight_layout()

## 4. Through the S3 bucket

The bucket is served by the local S3 service of grstbx (`grstbx-s3-bucket`, which serves all the buckets of
`/data/satellite/s3`; profile `grstbx-local`). The subsets are then read with s3fs, e.g. from another machine
or a JupyterHub session (the cell does nothing if the service is not running).

In [ ]:
import os
os.environ.setdefault('AWS_PROFILE', 'grstbx-local')
try:
    import s3fs
    fs = s3fs.S3FileSystem()
    url = row.href
    with fs.open(url) as f:
        remote = xr.open_dataset(f, engine='h5netcdf').load()
    print(url, dict(remote.sizes))
except Exception as exc:
    print(f'S3 service not reachable ({type(exc).__name__}): start it with grstbx-s3-bucket')